# Mars Arrival

Goals:
- Course corrections
- B-plane targeting to specific orbit
- Hyperbolic reentry orbit and burn
- Propagation

Idea (thanks Zeb)
- Same orbit but just RAAN change
- Inclination burn

**Assumptions**
- Circular target orbit

In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
import numpy as np
from cesium.constants import (
    DAY_TO_SEC,
    DEG_TO_RAD,
    HOUR_TO_SEC,
    KM_TO_M,
    M_TO_KM,
    MIN_TO_SEC,
    RAD_TO_DEG,
)
from cesium.guidance.arrival import b_plane_frame, plan_direct_injection
from cesium.guidance.ascent import ideal_ascent_dv, launch_azimuth
from cesium.guidance.interplanetary import (
    interplanetary_circ_injection,
)
from cesium.math.base import unit
from cesium.math.time import jd_to_greg
from cesium.orbital.coes import coes_to_rv, rv_to_coes
from cesium.orbital.interplanetary import hyperbolic_characteristics, sph_of_infl
from cesium.orbital.transformations import calc_dcm_mci_wrt_hci
from cesium.physics.dynamics import propagate_two_body
from cesium.plotting import plot_3d
from cesium.world.constants import MU_MARS, MU_SUN, R_MARS, W_MARS
from mission_io import export_fig, load_mission, save_mission
from numpy.lib.npyio import NpzFile
from numpy.linalg import norm

## Loading trajectory
From `interplanetary.ipynb`

In [24]:
MISSION_NAME = "birthday-christmas-earlier"
mission: NpzFile = load_mission("interplanetary", MISSION_NAME)


def print_mission(mission: NpzFile):
    """Print dates, duration, sampling, units, and stored fields from a mission archive.

    Parameters
    ----------
    mission : NpzFile
        Loaded Mars Courier .mission archive in NumPy NPZ format. Expected
        fields include lambert_departure_jd, lambert_arrival_jd, tof_s,
        time_s, dt_s, frame, position_units, velocity_units, and time_origin.
        Epochs are heliocentric Lambert references, not liftoff or SOI crossings.

    Notes
    -----
    Uses the notebook's MISSION_NAME as the display title. State-array shapes
    and dtypes are listed without changing or closing the loaded archive.
    """
    print(f"Mission: {MISSION_NAME}")
    print("Heliocentric reference times")
    for label, key in (
        ("Lambert departure (Earth reference)", "lambert_departure_jd"),
        ("Lambert arrival (Mars reference)", "lambert_arrival_jd"),
    ):
        print(
            f"{label}: "
            + "{:04.0f}-{:02.0f}-{:02.0f} {:02.0f}:{:02.0f}:{:06.3f}".format(
                *jd_to_greg(float(mission[key]))
            )
        )
    print(f"Flight duration: {float(mission['tof_s']) / 86400:.4f} days")
    print(
        f"Samples: {mission['time_s'].size} (seconds since {mission['time_origin'].item()})"
    )
    # print(f"Timesteps ranging from {mission['dt_s'].min():.3f}-{mission['dt_s'].max():.3f} s ")
    print(
        f"Frame: {mission['frame'].item()}; positions: {mission['position_units'].item()}; velocities: {mission['velocity_units'].item()}"
    )
    print("\nStored fields:")
    for key in mission.files:
        info = mission[key].dtype

        if mission[key].dtype != float:
            info = '"' + mission[key] + '"'
        elif mission[key].shape == ():
            info = mission[key]

        # info = mission[key].dtype if mission[key].dtype == float else mission[key]
        print(f"  {key:24s} {mission[key].shape!s:14s} {info}")


print_mission(mission)

lambert_arr_jd = float(mission["lambert_arrival_jd"])

# Mars-centered, then rotated into Mars-equatorial inertial (MCI) axes, fixed at arrival.
dcm_mci_wrt_hci = calc_dcm_mci_wrt_hci(lambert_arr_jd)
rv_sat_mars_hci_km = mission["satellite_rv_hci_km"] - mission["mars_rv_hci_km"]
rv_sat_mci_km = np.hstack(
    (
        rv_sat_mars_hci_km[:, 0:3] @ dcm_mci_wrt_hci.T,
        rv_sat_mars_hci_km[:, 3:6] @ dcm_mci_wrt_hci.T,
    )
)
r_lambert_end_mci_km = rv_sat_mci_km[-1, 0:3]
v_inf_mci_km = rv_sat_mci_km[-1, 3:6]

Mission: birthday-christmas-earlier
Heliocentric reference times
Lambert departure (Earth reference): 2035-06-14 12:00:00.000
Lambert arrival (Mars reference): 2035-12-25 00:00:00.000
Flight duration: 193.5000 days
Samples: 1936 (seconds since lambert_departure)
Frame: HCI; positions: km; velocities: km/s

Stored fields:
  stage                    ()             "interplanetary"
  mission_name             ()             "birthday_christmas_earlier"
  lambert_departure_jd     ()             2464493.0
  lambert_arrival_jd       ()             2464686.5
  tof_s                    ()             16718400.0
  time_s                   (1936,)        float64
  dt_s                     (1935,)        float64
  earth_rv_hci_km          (1936, 6)      float64
  mars_rv_hci_km           (1936, 6)      float64
  satellite_rv_hci_km      (1936, 6)      float64
  frame                    ()             "HCI"
  position_units           ()             "km"
  velocity_units           ()             "km

In [57]:
# Desired landing site, basically we want the orbit inclination to be above 18 deg
PERSEVERENCE_LLA = (18.434830485851624, 77.23277091979982, 0)

# Target (RPO) orbit in MCI: i and RAAN measured from Mars's equator
# a,e,i,raan,aop,ta
PARKING_ORBIT_ALT_M = 300e3  # m
PARKING_ORBIT_RAAN_DEG = 125
PARKING_ORBIT_I_DEG = 45
TARGET_COES = (
    R_MARS + PARKING_ORBIT_ALT_M,
    0.0,
    PARKING_ORBIT_I_DEG * DEG_TO_RAD,
    PARKING_ORBIT_RAAN_DEG * DEG_TO_RAD,
    0,
    0,
)

mars_soi_km = sph_of_infl(norm(mission["mars_rv_hci_km"][-1, 0:3]), MU_MARS, MU_SUN)

In [58]:
# Parking orbit

rv_target0 = np.concatenate(coes_to_rv(*TARGET_COES, MU_MARS))

t_target = np.linspace(0, 150 * MIN_TO_SEC, 10000)
rv_target_mci_m = propagate_two_body(rv_target0, t_target, MU_MARS)

## Arrival Characteristics

In [59]:
# Mars-centered, Mars-equatorial (MCI) axes.
C3_km2, rla, dla = hyperbolic_characteristics(v_inf_mci_km)

PERIAPSE = R_MARS + PARKING_ORBIT_ALT_M
dv_p, ta_asymptote, aiming_radius, e = interplanetary_circ_injection(
    norm(v_inf_mci_km * KM_TO_M), PERIAPSE, MU_MARS, deg=False
)

print("Arrival target (Lambert reference epoch)")
print(f"  v_inf: {norm(v_inf_mci_km):.4f} km/s; C3: {C3_km2:.4f} km²/s²")
print("  Incoming velocity asymptote angles in MCI (Mars-equatorial) axes")
print(
    f"  Right ascension: {rla * RAD_TO_DEG:.4f} deg; declination (DAP): {dla * RAD_TO_DEG:.4f} deg"
)
print("\nCircular parking orbit / ideal tangential capture burn")
print(f"  Parking altitude: {PARKING_ORBIT_ALT_M * M_TO_KM:.1f} km")
print(f"  Periapsis radius: {PERIAPSE * M_TO_KM:.3f} km from Mars's center")
print(f"  Circular speed: {np.sqrt(MU_MARS / PERIAPSE) * M_TO_KM:.4f} km/s")
print(
    f"  Pre-burn periapsis speed: {np.sqrt(norm(v_inf_mci_km * KM_TO_M) ** 2 + 2 * MU_MARS / PERIAPSE) * M_TO_KM:.4f} km/s"
)
print(f"  Required capture delta-v magnitude: {dv_p * M_TO_KM:.4f} km/s")
print("\nArrival hyperbola")
print(f"  Eccentricity: {e:.6f}")
print(f"  Incoming asymptotic true anomaly: {-ta_asymptote * RAD_TO_DEG:.4f} deg")
print(f"  Aiming radius (impact parameter): {aiming_radius * M_TO_KM:.3f} km")

Arrival target (Lambert reference epoch)
  v_inf: 2.8202 km/s; C3: 7.9534 km²/s²
  Incoming velocity asymptote angles in MCI (Mars-equatorial) axes
  Right ascension: 305.7635 deg; declination (DAP): 8.8954 deg

Circular parking orbit / ideal tangential capture burn
  Parking altitude: 300.0 km
  Periapsis radius: 3696.000 km from Mars's center
  Circular speed: 3.4041 km/s
  Pre-burn periapsis speed: 5.5793 km/s
  Required capture delta-v magnitude: 2.1752 km/s

Arrival hyperbola
  Eccentricity: 1.686364
  Incoming asymptotic true anomaly: -126.3696 deg
  Aiming radius (impact parameter): 7312.017 km


## Targeting Arrival

**Ellipse planner**
1. Arrival
   1. Target orbit
   2. Find B-plane aim point
2. Plane-change
   1. Rotate the plane by β at the node
3. Lower apoapsis

- Use B-plane targeting to choose, among the planes that contain Ŝ, the one closest to the target plane. Its angle to the target plane is β.
- At periapsis, burn tangentially into a high ellipse, for example a 1-sol orbit with r_a ≈ 37,200 km. There's no plane change in this burn.
- At the node with the target plane, ideally near apoapsis where the speed is about 456 m/s, rotate the plane by β.
- At periapsis, lower apoapsis to circularize at the target altitude.


**Direct Injection**
1. Small inclination or RAAN changes

Refs
- https://ai-solutions.com/_freeflyeruniversityguide/b_plane_targeting.htm (vectors)
- 

In [60]:
v_inf_mci = v_inf_mci_km * KM_TO_M

plan = plan_direct_injection(
    v_inf_mci,
    target_periapse_m=PERIAPSE,
    target_raan=PARKING_ORBIT_RAAN_DEG,
    target_i=PARKING_ORBIT_I_DEG,
    mu=MU_MARS,
    deg=True,
)
r_p_mci, v_p_mci = plan.r_p, plan.v_p_hyp
(capture_burn,) = plan.burns
dv_capture_mci = capture_burn.dv

## Validation


In [61]:
# Backwards from periapsis along the arrival hyperbola (no velocity flip needed),
# then reversed so time runs forward: -12 h ... 0 (periapsis)
t_traj_s = np.linspace(0, -12 * HOUR_TO_SEC, 100000)
rv_traj_mci_m = propagate_two_body(
    np.concatenate((r_p_mci, v_p_mci)), t_traj_s, MU_MARS
)
t_traj_s, rv_traj_mci_m = t_traj_s[::-1], rv_traj_mci_m[::-1]

print("Direct injection")
print(
    f"  beta (asymptote to target plane): {plan.beta * RAD_TO_DEG:.2f} deg "
    f"(limit arccos(1/e) = {np.arccos(1 / e) * RAD_TO_DEG:.2f} deg)"
)
print(
    f"  Aim point: B.T = {plan.B_TR[0] * M_TO_KM:.1f} km, B.R = {plan.B_TR[1] * M_TO_KM:.1f} km"
)
print(
    f"  Capture + plane change dv: {plan.dv_total * M_TO_KM:.4f} km/s "
    f"(coplanar minimum {dv_p * M_TO_KM:.4f} km/s)"
)

# Validation
rv_far = propagate_two_body(
    np.concatenate((r_p_mci, v_p_mci)),
    [0.0, -200 * DAY_TO_SEC],
    MU_MARS,
    rtol=1e-12,
    atol=1e-6,
)[-1]
asymptote_err = np.arccos(np.clip(unit(rv_far[3:]) @ unit(v_inf_mci), -1, 1))
_, e_cap, i_cap, raan_cap, _, _ = rv_to_coes(r_p_mci, plan.v_p_capture, MU_MARS)
print("\nValidation")
print(f"  Asymptote vs v_inf: {asymptote_err * RAD_TO_DEG:.4f} deg")
print(f"  |r_p| - target: {norm(r_p_mci) - PERIAPSE:.3e} m")
print(
    f"  b_plane_frame vs plan: {norm(b_plane_frame(r_p_mci, v_p_mci, MU_MARS) - plan.B_TR):.3e} m"
)
print(
    f"  Post-burn orbit: e = {e_cap:.2e}, i = {i_cap * RAD_TO_DEG:.6f} deg, "
    f"RAAN = {raan_cap * RAD_TO_DEG:.6f} deg"
)

Direct injection
  beta (asymptote to target plane): 6.81 deg (limit arccos(1/e) = 53.63 deg)
  Aim point: B.T = 5572.1 km, B.R = -4734.7 km
  Capture + plane change dv: 2.2685 km/s (coplanar minimum 2.1752 km/s)

Validation
  Asymptote vs v_inf: 0.0000 deg
  |r_p| - target: 0.000e+00 m
  b_plane_frame vs plan: 0.000e+00 m
  Post-burn orbit: e = 3.24e-16, i = 45.000000 deg, RAAN = 125.000000 deg


### Descent from parking orbit (ideal)

Minimum Δv between the Perseverance landing site and the 300 km, i = 45° parking orbit,
with no perturbations and Mars rotation assistance

In [62]:
site_lat = PERSEVERENCE_LLA[0] * DEG_TO_RAD
r_site_mars = R_MARS + PERSEVERENCE_LLA[2]
v_east_mars = W_MARS * r_site_mars * np.cos(site_lat)
mars_az = launch_azimuth(site_lat, PARKING_ORBIT_I_DEG * DEG_TO_RAD)

mars_surface = ideal_ascent_dv(r_site_mars, PERIAPSE, v_east_mars, mars_az, MU_MARS)

print(
    f"Ideal Mars surface <-> parking orbit: {PERSEVERENCE_LLA[0]:.2f} N -> i = {PARKING_ORBIT_I_DEG} deg, 300 km"
)
print(
    f"  Eastward rotation speed {v_east_mars:.1f} m/s; azimuth {mars_az * RAD_TO_DEG:.2f} deg from north"
)
print(
    f"  Burn at surface: {mars_surface['dv_burn1']:.1f} m/s; circularize: {mars_surface['dv_circularize']:.1f} m/s"
)
print(
    f"  Total: {mars_surface['dv_total']:.1f} m/s (rotation gain {mars_surface['rotation_gain']:.1f} m/s; "
    f"non-rotating {mars_surface['dv_no_rotation']:.1f} m/s)"
)

Ideal Mars surface <-> parking orbit: 18.43 N -> i = 45 deg, 300 km
  Eastward rotation speed 228.4 m/s; azimuth 48.19 deg from north
  Burn at surface: 3458.7 m/s; circularize: 72.8 m/s
  Total: 3531.5 m/s (rotation gain 166.9 m/s; non-rotating 3698.3 m/s)


**TODO: high elliptical orbit with plane change**

## Visualization
$
\boxed{
\hat{\mathbf p}
=\frac{1}{eq}\hat{\mathbf u}
\ \pm\
\sqrt{1-\frac{1}{e^2q^2}}\,
(\hat{\mathbf n}\times\hat{\mathbf u})
}$


$
\hat s=\frac{\hat p}{e}+\sqrt{1-\frac1{e^2}}\,\hat t
$

In [65]:
# Sun direction at arrival, rotated into MCI.
sun_direction_hci = dcm_mci_wrt_hci @ -mission["mars_rv_hci_km"][-1, :3]


frames = [
    # {
    #     "name": "HCI",
    #     "location": [0, 0, 0],
    #     "dcm_body_wrt_hci": np.eye(3),
    #     "length": 10000,
    # },
]

vectors = [
    {
        "direction": sun_direction_hci,
        "length": 20000,
        "name": "Sun direction",
        "color": "goldenrod",
        "label": True,
    },
    {
        "direction": dv_capture_mci,
        "length": 10000,
        "name": "dv direction",
        "color": "black",
        "label": True,
        "origin": r_p_mci * M_TO_KM,
    },  # just for visualization
    # {
    #     "direction": arrival.v_inf_hci_km,
    #     "length": 10000,
    #     "name": "v_vec",
    #     "color": "orange",
    #     "label": True,
    # },
    # {
    #     "direction": arrival.orbit_right_vec,
    #     "length": 10000,
    #     "name": "orbit right vec",
    #     "color": "black",
    #     "label": True,
    # },  # just for visualization
    # {
    #     "direction": arrival.orbit_up_vec,
    #     "length": 10000,
    #     "name": "orbit up vec",
    #     "color": "black",
    #     "label": True,
    # },  # just for visualization
    # {
    #     "direction": arrival.orbit_normal,
    #     "length": 10000,
    #     "name": "orbit normal vec",
    #     "color": "black",
    #     "label": True,
    # },  # just for visualization
    # {
    #     "direction": arrival.v_capture_mci_m_s,
    #     "length": 10000,
    #     "name": "V capture direction vec",
    #     "color": "black",
    #     "label": True,
    #     "origin": arrival.r_p_mci_m * M_TO_KM
    # },  # just for visualization
    # {
    #     "direction": arrival.v_p_mci_m_s,
    #     "length": 10000,
    #     "name": "V capture direction vec",
    #     "color": "black",
    #     "label": True,
    #     "origin": arrival.r_p_mci_m * M_TO_KM
    # },  # just for visualization
]

points = [
    {
        "name": "Periapse",
        "location": (r_p_mci) * M_TO_KM,
        "color": "red",
    },
    # {
    #     "name": "SOI along v_inf",
    #     "location": -mars_soi_km * unit(v_inf_mci_km),
    #     "color": "green",
    # },
]

view_range = 30000
fig_earth = plot_3d(
    [
        rv_sat_mci_km[:, 0:3],
        rv_target_mci_m[:, 0:3] * M_TO_KM,
        rv_traj_mci_m[:, 0:3] * M_TO_KM,
    ],
    [mission["time_s"], t_target, t_traj_s],
    labels=["Lambert Arrival Direction", "Target Orbit", "Propagated from V_inf"],
    trace_colors=["purple", "black", "green"],
    bodies={"name": "Mars", "radius": R_MARS * M_TO_KM, "color": "orangered"},
    frames=frames,
    vectors=vectors,
    points=points,
    title=f"Mars arrival — {view_range} km view",
    units="km",
    azim=80,
    elev=30,
    show_end=False,
    show=False,
)
fig_earth.update_scenes(
    xaxis_range=[-view_range, view_range],  # km
    yaxis_range=[-view_range, view_range],  # km
    zaxis_range=[-view_range, view_range],  # km
)
fig_earth.show()

In [ ]:
export_fig(fig_earth, "arrival", "arrival_hyperbola_mci")

## Save arrival

Export for `rpo.ipynb` and `summary.ipynb`. t = 0 is periapsis passage and the
capture burn, placed at the Lambert arrival epoch (assumed TDB). States are MCI
(Mars-equatorial inertial), meters and m/s.

In [64]:
rv_capture_mci_m = np.concatenate((r_p_mci, plan.v_p_capture))
assert np.isfinite(rv_traj_mci_m).all() and np.isfinite(rv_capture_mci_m).all()
arrival_path = save_mission(
    "arrival",
    MISSION_NAME,
    epoch_jd=lambert_arr_jd,
    # Approach hyperbola, -12 h ... 0 (periapsis)
    time_s=t_traj_s,
    rv_mci_m=rv_traj_mci_m,
    # Post-capture state at t = 0: the RPO chaser's nominal initial state
    rv_capture_mci_m=rv_capture_mci_m,
    target_coes=np.array(TARGET_COES, dtype=float),  # a [m], e, i, raan, aop, ta [rad]
    v_inf_mci_km_s=v_inf_mci_km,
    B_TR_m=plan.B_TR,
    beta_rad=plan.beta,
    # Delta-v budget
    dv_capture_m_s=plan.dv_total,
    dv_capture_mci_m_s=dv_capture_mci,
    dv_coplanar_min_m_s=dv_p,
    capture_strategy="direct_injection",
    dv_surface_ideal_m_s=mars_surface[
        "dv_total"
    ],  # surface <-> parking orbit, lossless
    landing_site_lla_deg_m=np.array(PERSEVERENCE_LLA, dtype=float),
    frame="MCI",
    position_units="m",
    velocity_units="m/s",
    time_origin="periapsis_capture_burn",
    time_units="s",
)
print(arrival_path)

/Users/cskin/cesium/gnc/projects/mars-courier/trajectories/arrival-birthday_christmas_earlier.mission
